# Extract hand landmarks

This notebook reads the selected sign folders, finds hand landmarks with MediaPipe, and saves numbers for model training.

The selected folders are listed in `KEEP`. Folder names must match exactly.

**Output in the `models/sign_classifier/results/` folder**

- `landmarks.csv`: one row per picture with the sign name, picture path, and 129 numbers (`f0` to `f128`).
- `failed_pictures.csv`: the pictures where no hand was found.

**The 129 numbers for each picture**

- 0 to 62: first hand, 21 points x (x, y, z).
- 63 to 125: second hand, with zeros if the sign uses one hand.
- 126 to 128: distance between the two wrists (x, y, z).

The points are made relative to the wrist and divided by the hand size. The numbers describe the **shape** of the hand, not its position in the picture or its distance from the camera.


## 1. Imports and project paths

The dataset must contain a folder named `Data`. It can be under `assets/sign_data/Data`.

In [5]:
from pathlib import Path
import csv
import cv2
import urllib.request
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

NOTEBOOK_DIR = Path.cwd()
PROJECT = NOTEBOOK_DIR
while PROJECT != PROJECT.parent and not (PROJECT / "models" / "sign_classifier").exists():
    PROJECT = PROJECT.parent

# Find the dataset folder named Data in supported project locations.
search_dirs = [
    PROJECT / "assets" / "sign_data" / "(First ever) Tunisian Sign Language Dataset" / "Data",
    PROJECT / "assets" / "sign_data",
    PROJECT / "data" / "raw",
    PROJECT / "models" / "sign_classifier" / "exploration"
]

found = []
for search_dir in search_dirs:
    if search_dir.is_dir():
        if search_dir.name == "Data":
            found.append(search_dir)
            break
        sub_dirs = [path for path in search_dir.rglob("Data") if path.is_dir()]
        if sub_dirs:
            found.extend(sub_dirs)
            break

if not found:
    raise SystemExit("Folder 'Data' not found. Put the dataset under assets/sign_data/, data/raw/ or models/sign_classifier/exploration/")
ROOT = found[0]

KEEP = [
    "3aslema", "labes", "oui", "non", "lyoum",
    "baladya", "banka", "sbitar", "dar",
    "taxi", "louage", "jom3a", "om", "enti", "n3awnek", "ma7kma", "5mis", "a7ad", "erb3a", "sebt", "thleth", "thnin", "karhba", "car"
]

print("Dataset folder:", ROOT)
print("Signs to keep:", KEEP)

# MediaPipe uses the Tasks API and needs this hand model file.
MODEL_DIR = PROJECT / "assets" / "sign_data"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "hand_landmarker.task"
if not MODEL_PATH.exists():
    alt_path = PROJECT / "models" / "hand_landmarker.task"
    if alt_path.exists():
        MODEL_PATH = alt_path

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
if not MODEL_PATH.exists():
    print("Downloading the MediaPipe hand model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Hand model:", MODEL_PATH)

Dataset folder: c:\Users\minya\Desktop\tunisian-sign-language-to-speech\assets\sign_data\(First ever) Tunisian Sign Language Dataset\Data
Signs to keep: ['3aslema', 'labes', 'oui', 'non', 'lyoum', 'baladya', 'banka', 'sbitar', 'dar', 'taxi', 'louage', 'jom3a', 'om', 'enti', 'n3awnek', 'ma7kma', '5mis', 'a7ad', 'erb3a', 'sebt', 'thleth', 'thnin', 'karhba', 'car']
Hand model: c:\Users\minya\Desktop\tunisian-sign-language-to-speech\assets\sign_data\hand_landmarker.task


## 2. Find landmarks for one picture

MediaPipe returns points for each detected hand. We move the points so the wrist is at zero. We also divide by hand size. This keeps the hand shape and reduces the effect of camera distance.

In [6]:
def get_features(landmarker, path):
    """Return 129 numbers for one picture, or None if no hand is found."""
    img = cv2.imread(path)
    if img is None:
        return None
    image = mp.Image.create_from_file(path)
    result = landmarker.detect(image)
    if not result.hand_landmarks:
        return None

    found_hands = []
    for hand in result.hand_landmarks:
        pts = np.array([[p.x, p.y, p.z] for p in hand])
        wrist = pts[0].copy()
        pts = pts - wrist  # make points relative to the wrist
        size = np.linalg.norm(pts[9]) + 1e-6  # hand size: wrist to middle finger base
        found_hands.append((wrist, (pts / size).flatten()))
    found_hands.sort(key=lambda h: h[0][0])  # left-most hand first

    feats = np.zeros(129)
    feats[:63] = found_hands[0][1]  # first hand: 63 numbers
    if len(found_hands) >= 2:
        feats[63:126] = found_hands[1][1]  # second hand: 63 numbers
        feats[126:129] = found_hands[1][0] - found_hands[0][0]  # distance between the wrists
    return feats

## 3. Select pictures

Only pictures whose sign folder is in `KEEP` will be processed. Other dataset folders stay untouched.

In [7]:
# List the pictures of the signs we keep.
pictures = []
folders_seen = set()
for img in ROOT.rglob("*"):
    if img.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        category, sign = img.relative_to(ROOT).parts[:2]
        folders_seen.add(sign)
        if sign in KEEP:
            pictures.append((sign, str(img)))

# Safety check: warn about names in KEEP that do not exist in the dataset.
missing = [s for s in KEEP if s not in folders_seen]
if missing:
    print("WARNING: these names in KEEP match no folder:", missing)
print(len(pictures), "pictures to process")

1409 pictures to process


## 4. Extract and save the numbers

The notebook saves one row per picture in `models/sign_classifier/results/landmarks.csv`. Pictures with no detected hand go to `models/sign_classifier/results/failed_pictures.csv`.

In [8]:
# Run MediaPipe on every picture.
rows, failed = [], []
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
)
with vision.HandLandmarker.create_from_options(options) as hands:
    for i, (sign, path) in enumerate(pictures, 1):
        f = get_features(hands, path)
        if f is None:
            failed.append((sign, path))
        else:
            rows.append([sign, path, *f])
        if i % 100 == 0:
            print(i, "done")

out = PROJECT / "models" / "sign_classifier" / "results"
out.mkdir(exist_ok=True)
cols = ["sign", "path"] + [f"f{i}" for i in range(129)]
with (out / "landmarks.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(cols)
    writer.writerows(rows)
with (out / "failed_pictures.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["sign", "path"])
    writer.writerows(failed)

print("Saved in", out)

100 done
200 done
300 done
400 done
500 done
600 done
700 done
800 done
900 done
1000 done
1100 done
1200 done
1300 done
1400 done
Saved in c:\Users\minya\Desktop\tunisian-sign-language-to-speech\models\sign_classifier\results


## 5. Check the detection rate

A sign with a detection rate under 70 percent should be dropped from the training set.

In [9]:
# Detection rate per sign: percent of pictures where a hand was found.
total = {sign: sum(1 for picture_sign, _ in pictures if picture_sign == sign) for sign in KEEP}
ok = {sign: sum(1 for row in rows if row[0] == sign) for sign in KEEP}
rate = {sign: round(ok[sign] / total[sign] * 100, 1) if total[sign] else 0.0 for sign in KEEP}
for sign in sorted(rate, key=rate.get):
    print(f"{sign}: {rate[sign]}%")

karhba: 44.4%
banka: 72.4%
thnin: 76.6%
erb3a: 77.3%
louage: 78.9%
non: 84.6%
dar: 92.4%
sebt: 94.4%
oui: 94.7%
ma7kma: 95.5%
thleth: 95.5%
car: 97.0%
sbitar: 97.9%
lyoum: 98.0%
5mis: 98.1%
3aslema: 100.0%
labes: 100.0%
baladya: 100.0%
taxi: 100.0%
jom3a: 100.0%
om: 100.0%
enti: 100.0%
n3awnek: 100.0%
a7ad: 100.0%
